In [1]:
# 1. Clone repository & install ONLY missing helper packages
!git clone https://github.com/ahsan-c0ding/S4-Enhancement-Exploration.git
%cd S4-Enhancement-Exploration
!git checkout python

import os
import sys
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# Install lightweight missing packages
%pip install einops coloredlogs torchinfo --quiet
%pip install git+https://github.com/mwalmsley/galaxy_mnist.git@c1fe9853a00bc34b2ff082585c6bb1654d34d239 --quiet

# Symlink workaround for model_params path resolving
parent_dir = os.path.dirname(current_dir)
_shim_path = os.path.join(parent_dir, "model_params")
_real_path = os.path.join(current_dir, "model_params")
if not os.path.exists(_shim_path) and os.path.exists(_real_path):
    os.symlink(_real_path, _shim_path)

Cloning into 'S4-Enhancement-Exploration'...
remote: Enumerating objects: 1152, done.
remote: Counting objects: 100% (220/220), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 1152 (delta 88), reused 180 (delta 68), pack-reused 932 (from 2)
Receiving objects: 100% (1152/1152), 94.54 MiB | 43.90 MiB/s, done.
Resolving deltas: 100% (489/489), done.
/kaggle/working/S4-Enhancement-Exploration
Branch 'python' set up to track remote branch 'python' from 'origin'.
Switched to a new branch 'python'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
  Preparing metadata (setup.py) ... done
Note: you may need to restart the kernel to use updated packages.


In [2]:
import math
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from einops import repeat

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, precision_score, recall_score, roc_auc_score
from torchinfo import summary

from model.functions import load_data

warnings.filterwarnings("ignore")
sns.set_style("darkgrid")
plt.rcParams["figure.figsize"] = [11, 6]

# Reproducibility
RNG_SEED = 30485
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(RNG_SEED)

CLASS_NAMES = ["Smooth Round", "Smooth Cigar", "Edge-on Disk", "Unbarred Spiral"]

# ============================================================================
# HYPERPARAMETERS: TARGETING 60-80K PARAMS FOR LINEAR + S4D
# ============================================================================
COLORED = True
IN_CHANNELS = 3 if COLORED else 1

S4D_NUM_LAYERS = 3
S4D_PATCH_SIZE = 4
S4D_POOLING = "last"
S4D_USE_NORM = False
S4D_USE_RESIDUAL = False
S4D_DROPOUT = 0.2

S4D_BATCH_SIZE = 32
S4D_FINAL_EPOCHS = 630
LEARNING_RATE = 1e-3

# Increasing D from 72 to 108 to hit ~76k parameters without a Conv Stem
TARGET_D = 108 

print(f"Device: {DEVICE}")
print(f"Target Model: Linear Embed + S4D | layers={S4D_NUM_LAYERS} | d_model={TARGET_D} | pooling={S4D_POOLING}")

/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Device: cuda
Target Model: Linear Embed + S4D | layers=3 | d_model=108 | pooling=last


In [3]:
class HilbertScan(nn.Module):
    def __init__(self, image_size=64, patch_size=1):
        super().__init__()
        self.image_size = image_size
        self.patch_size = patch_size
        self.grid_size = image_size // patch_size
        self.num_patches = self.grid_size ** 2
        self.register_buffer("indices", self._get_hilbert_indices(self.grid_size))

    @staticmethod
    def _rot(s, x, y, rx, ry):
        if ry == 0:
            if rx == 1:
                x = s - 1 - x
                y = s - 1 - y
            x, y = y, x
        return x, y

    def _d2xy(self, n, d):
        x = y = 0
        t, s = d, 1
        while s < n:
            rx = (t // 2) & 1
            ry = (t ^ rx) & 1
            x, y = self._rot(s, x, y, rx, ry)
            x += s * rx
            y += s * ry
            t //= 4
            s *= 2
        return x, y

    def _get_hilbert_indices(self, grid_size):
        indices = []
        for d in range(grid_size * grid_size):
            x, y = self._d2xy(grid_size, d)
            indices.append(y * grid_size + x)
        return torch.LongTensor(indices)

    def forward(self, x):
        B, C, H, W = x.shape
        p = self.patch_size
        patches = x.unfold(2, p, p).unfold(3, p, p)
        patches = patches.permute(0, 2, 3, 1, 4, 5).contiguous()
        patches = patches.view(B, self.num_patches, C * p * p)
        return patches[:, self.indices, :]

class TakeLastTimestep(nn.Module):
    def forward(self, x):
        return x[:, -1, :]

class S4DConv(nn.Module):
    def __init__(self, d_model, d_state=64, dt_min=0.001, dt_max=0.1, transposed=True, lr=None):
        super().__init__()
        self.h = d_model
        self.n = d_state
        self.transposed = transposed

        log_dt = torch.rand(self.h) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        log_A_real = torch.log(0.5 * torch.ones(self.h, self.n // 2))
        A_imag = math.pi * repeat(torch.arange(self.n // 2), 'n -> h n', h=self.h)
        C_init = torch.randn(self.h, self.n // 2, dtype=torch.cfloat)

        self.register("log_dt", log_dt, lr)
        self.register("log_A_real", log_A_real, lr)
        self.register("A_imag", A_imag, lr)

        self.C = nn.Parameter(torch.view_as_real(C_init))
        self.D = nn.Parameter(torch.randn(self.h))

    def register(self, name, tensor, lr=None):
        if lr == 0.0:
            self.register_buffer(name, tensor)
        else:
            self.register_parameter(name, nn.Parameter(tensor))
            optim = {"weight_decay": 0.0}
            if lr is not None:
                optim["lr"] = lr
            setattr(getattr(self, name), "_optim", optim)

    def forward(self, u):
        if not self.transposed:
            u = u.transpose(-1, -2)
        L = u.size(-1)

        dt = torch.exp(self.log_dt)
        C = torch.view_as_complex(self.C)
        A = -torch.exp(self.log_A_real) + 1j * self.A_imag

        dtA = A * dt.unsqueeze(-1)
        K_exp = torch.exp(dtA.unsqueeze(-1) * torch.arange(L, device=u.device))
        C_tilde = C * (torch.exp(dtA) - 1.) / A
        k = 2 * torch.einsum('hn, hnl -> hl', C_tilde, K_exp).real

        k_f = torch.fft.rfft(k, n=2 * L)
        u_f = torch.fft.rfft(u, n=2 * L)
        y = torch.fft.irfft(u_f * k_f, n=2 * L)[..., :L]
        y = y + u * self.D.unsqueeze(-1)

        if not self.transposed:
            y = y.transpose(-1, -2)
        return y, None

class GalaxyClassifierS4D(nn.Module):
    def __init__(self, d_model=108, num_classes=4, num_layers=3, patch_size=4):
        super().__init__()
        self.patch_size = patch_size
        self.hilbert_scan = HilbertScan(image_size=64, patch_size=patch_size)
        
        # Pure Linear Embedding
        patch_dim = 3 * patch_size * patch_size
        self.uproject = nn.Linear(patch_dim, d_model)
        
        self.s4_layers = nn.ModuleList([
            S4DConv(d_model=d_model, d_state=d_model, transposed=False)
            for _ in range(num_layers)
        ])
        self.acts = nn.ModuleList([nn.GELU() for _ in range(num_layers)])
        self.drop = nn.Dropout(S4D_DROPOUT)
        self.take_last = TakeLastTimestep()
        
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x, return_logits=True):
        x_seq = self.hilbert_scan(x)
        h = self.uproject(x_seq)

        for s4_layer, act in zip(self.s4_layers, self.acts):
            h_out, _ = s4_layer(h)
            h_out = act(h_out)
            h = self.drop(h_out)

        pooled = self.take_last(h)
        logits = self.fc(pooled)

        if return_logits:
            return logits
        return nn.Softmax(dim=-1)(logits)

In [4]:
# Load dataset
X, y_onehot, y = load_data(root="./data", download=True, train=True, colored=COLORED)
X_test, y_test_onehot, y_test = load_data(root="./data", download=True, train=False, colored=COLORED)
NUM_CLASSES = y_onehot.shape[1]

x_train, x_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RNG_SEED, stratify=y
)

class GalaxyDataset(Dataset):
    def __init__(self, images, labels, augment=False):
        self.images = images
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        lbl = self.labels[idx]
        if self.augment:
            k = random.randint(0, 3)
            img = torch.rot90(img, k, [1, 2])
            if random.random() > 0.5:
                img = torch.flip(img, [2])
            if random.random() > 0.5:
                img = torch.flip(img, [1])
        return img, lbl

train_loader = DataLoader(GalaxyDataset(x_train, y_train, augment=True), batch_size=S4D_BATCH_SIZE, shuffle=True)
val_loader = DataLoader(GalaxyDataset(x_val, y_val, augment=False), batch_size=S4D_BATCH_SIZE, shuffle=False)
test_loader = DataLoader(GalaxyDataset(X_test, y_test, augment=False), batch_size=S4D_BATCH_SIZE, shuffle=False)

print(f"Data Split -> Train: {len(x_train)} | Val: {len(x_val)} | Test: {len(X_test)}")

100%|██████████| 68.7M/68.7M [00:05<00:00, 12.8MB/s]
100%|██████████| 17.3M/17.3M [00:13<00:00, 1.28MB/s]


Original Dataset Size: 8000 samples
Original Dataset Size: 2000 samples
Data Split -> Train: 6400 | Val: 1600 | Test: 2000


In [5]:
def create_s4d_optimizer(model, lr=1e-3):
    decay_params, no_decay_params, special_params = [], [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if hasattr(param, "_optim"):
            special_params.append({'params': [param], 'lr': param._optim.get("lr", lr), 'weight_decay': 0.0})
        elif any(k in name for k in ["bias", "norm"]):
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    groups = [
        {'params': decay_params, 'weight_decay': 0.01, 'lr': lr},
        {'params': no_decay_params, 'weight_decay': 0.0, 'lr': lr},
    ] + special_params
    return torch.optim.AdamW(groups)

model = GalaxyClassifierS4D(d_model=TARGET_D, num_classes=NUM_CLASSES, num_layers=S4D_NUM_LAYERS, patch_size=S4D_PATCH_SIZE).to(DEVICE)
optimizer = create_s4d_optimizer(model, lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-5)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
os.makedirs("checkpoints", exist_ok=True)
ckpt_path = "checkpoints/target_linear_s4d.pt"

print(f"Starting Training for {S4D_FINAL_EPOCHS} epochs...")
for epoch in range(S4D_FINAL_EPOCHS):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
    scheduler.step()

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            val_correct += (model(images).argmax(-1) == labels).sum().item()
            val_total += labels.size(0)
    
    val_acc = val_correct / val_total
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), ckpt_path)

    if (epoch + 1) % 50 == 0 or epoch == S4D_FINAL_EPOCHS - 1:
        print(f"Epoch {epoch+1:03d}/{S4D_FINAL_EPOCHS} | Val Acc: {val_acc:.4f} (Best: {best_val_acc:.4f})")

Starting Training for 630 epochs...
Epoch 050/630 | Val Acc: 0.7594 (Best: 0.7594)
Epoch 100/630 | Val Acc: 0.7738 (Best: 0.7844)
Epoch 150/630 | Val Acc: 0.7794 (Best: 0.7863)
Epoch 200/630 | Val Acc: 0.7875 (Best: 0.7913)
Epoch 250/630 | Val Acc: 0.7869 (Best: 0.7956)
Epoch 300/630 | Val Acc: 0.7919 (Best: 0.7975)
Epoch 350/630 | Val Acc: 0.7875 (Best: 0.8006)
Epoch 400/630 | Val Acc: 0.7913 (Best: 0.8006)
Epoch 450/630 | Val Acc: 0.7931 (Best: 0.8056)
Epoch 500/630 | Val Acc: 0.8000 (Best: 0.8056)
Epoch 550/630 | Val Acc: 0.7944 (Best: 0.8056)
Epoch 600/630 | Val Acc: 0.7931 (Best: 0.8056)
Epoch 630/630 | Val Acc: 0.7950 (Best: 0.8056)


In [6]:
model.load_state_dict(torch.load(ckpt_path))
model.eval()

all_preds, all_targets, all_probs = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        probs = model(images.to(DEVICE), return_logits=False)
        all_preds.extend(probs.argmax(-1).cpu().numpy())
        all_targets.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_probs = np.array(all_probs)

print("=" * 50)
print("Target Linear+S4D (~76K) -- Test Set Results")
print("=" * 50)
print(f"Accuracy:        {accuracy_score(all_targets, all_preds) * 100:.2f}%")
print(f"Macro-F1:        {f1_score(all_targets, all_preds, average='macro'):.4f}")
print(f"Macro-Precision: {precision_score(all_targets, all_preds, average='macro'):.4f}")
print(f"Macro-Recall:    {recall_score(all_targets, all_preds, average='macro'):.4f}")
print(f"Macro-AUC (OVR): {roc_auc_score(all_targets, all_probs, multi_class='ovr', average='macro'):.4f}")
print("=" * 50)

print("\nModel Parameter Count:")
param_count = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {param_count:,}")

print("\nFull layer-by-layer breakdown:")
summary(model, input_size=(S4D_BATCH_SIZE, IN_CHANNELS, 64, 64))

Target Linear+S4D (~76K) -- Test Set Results
Accuracy:        78.95%
Macro-F1:        0.7902
Macro-Precision: 0.7907
Macro-Recall:    0.7903
Macro-AUC (OVR): 0.9551

Model Parameter Count:
Total Parameters: 76,360

Full layer-by-layer breakdown:


Layer (type:depth-idx)                   Output Shape              Param #
GalaxyClassifierS4D                      [32, 4]                   --
├─HilbertScan: 1-1                       [32, 256, 48]             --
├─Linear: 1-2                            [32, 256, 108]            5,292
├─ModuleList: 1-9                        --                        (recursive)
│    └─S4DConv: 2-1                      [32, 256, 108]            23,544
├─ModuleList: 1-10                       --                        --
│    └─GELU: 2-2                         [32, 256, 108]            --
├─Dropout: 1-5                           [32, 256, 108]            --
├─ModuleList: 1-9                        --                        (recursive)
│    └─S4DConv: 2-3                      [32, 256, 108]            23,544
├─ModuleList: 1-10                       --                        --
│    └─GELU: 2-4                         [32, 256, 108]            --
├─Dropout: 1-8                           [32, 256, 108] 